# 02 — Build box scores

Build box-score files for all 1,230 games in nba_2025_26_odds_scores.csv.

Source: Basketball-Reference box score pages. URL mapping was built from the
already-downloaded monthly schedule pages (bref/*.html) and saved to
boxscore_urls.json; every odds game matched (1,230/1,230).

Outputs:
  boxscores_player.csv — one row per player-game (incl. DNP rows, mp="DNP")
  boxscores_team.csv   — one row per team-game (2,460 rows), with officials

Checkpointing: per-month part files in boxscores_parts/; re-running resumes
from completed months. Failures logged to failures_boxscores.json.
Fetch: polite ~2s delay, browser UA, one retry. No login.


In [ ]:
import json
import os
import re
import time

import pandas as pd

from pathlib import Path
# project root: works whether this runs from notebooks/ or the project root
def _find_proj(start):
    p = Path(start).resolve()
    for c in [p, *p.parents]:
        if (c / 'data' / 'raw' / 'nba_2025_26_odds_scores.csv').exists():
            return c
        if (c / 'data' / 'raw').is_dir() and (c / 'notebooks').is_dir():
            return c
    return p
PROJ = _find_proj(Path.cwd())
RAW = PROJ / 'data' / 'raw'
INTERIM = PROJ / 'data' / 'interim'
OUT = PROJ / 'outputs'
RES = PROJ / 'results'
CACHE = PROJ / 'cache'

ODDS = RAW / 'nba_2025_26_odds_scores.csv'
URLMAP = CACHE / 'boxscore_urls.json'
PARTS = INTERIM / 'boxscores_parts'
PARTS.mkdir(parents=True, exist_ok=True)
FAIL_LOG = CACHE / 'failures_boxscores.json'

UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"}
DELAY = 2.0

BR_CODE = {'Atlanta Hawks': 'ATL', 'Boston Celtics': 'BOS', 'Brooklyn Nets': 'BRK',
           'Charlotte Hornets': 'CHO', 'Chicago Bulls': 'CHI', 'Cleveland Cavaliers': 'CLE',
           'Dallas Mavericks': 'DAL', 'Denver Nuggets': 'DEN', 'Detroit Pistons': 'DET',
           'Golden State Warriors': 'GSW', 'Houston Rockets': 'HOU', 'Indiana Pacers': 'IND',
           'Los Angeles Clippers': 'LAC', 'Los Angeles Lakers': 'LAL', 'Memphis Grizzlies': 'MEM',
           'Miami Heat': 'MIA', 'Milwaukee Bucks': 'MIL', 'Minnesota Timberwolves': 'MIN',
           'New Orleans Pelicans': 'NOP', 'New York Knicks': 'NYK', 'Oklahoma City Thunder': 'OKC',
           'Orlando Magic': 'ORL', 'Philadelphia 76ers': 'PHI', 'Phoenix Suns': 'PHO',
           'Portland Trail Blazers': 'POR', 'Sacramento Kings': 'SAC', 'San Antonio Spurs': 'SAS',
           'Toronto Raptors': 'TOR', 'Utah Jazz': 'UTA', 'Washington Wizards': 'WAS'}

PLAYER_COLS = ["date", "away_team", "home_team", "team", "player", "mp",
               "fg", "fga", "fg_pct", "fg3", "fg3a", "ft", "fta",
               "orb", "drb", "trb", "ast", "stl", "blk", "tov", "pf", "pts", "plus_minus"]
TEAM_COLS = ["date", "away_team", "home_team", "team", "mp",
             "fg", "fga", "fg_pct", "fg3", "fg3a", "ft", "fta",
             "orb", "drb", "trb", "ast", "stl", "blk", "tov", "pf", "pts", "plus_minus",
             "officials"]

STAT_MAP = {"MP": "mp", "FG": "fg", "FGA": "fga", "FG%": "fg_pct", "3P": "fg3",
            "3PA": "fg3a", "FT": "ft", "FTA": "fta", "ORB": "orb", "DRB": "drb",
            "TRB": "trb", "AST": "ast", "STL": "stl", "BLK": "blk", "TOV": "tov",
            "PF": "pf", "PTS": "pts", "+/-": "plus_minus"}


In [ ]:
def fetch(url, timeout=30):
    import urllib.request
    last = None
    for attempt, wait in enumerate((0, 5)):
        if wait:
            time.sleep(wait)
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read().decode("utf-8", "replace")
        except Exception as e:
            last = e
            print(f"  retry {attempt + 1}/2 {url} {e!r}", flush=True)
    raise last


def parse_officials(html):
    m = re.search(r"Officials:&nbsp;</strong>(.*?)</div>", html, re.S)
    if not m:
        return ""
    names = re.findall(r">([^<>]+)</a>", m.group(1))
    return "; ".join(n.strip() for n in names)


def parse_table(html, table_id):
    """Return (player_rows, team_row_dict) for one box-XXX-game-basic table."""
    dfs = pd.read_html(html, attrs={"id": table_id})
    df = dfs[0]
    # flatten MultiIndex columns: ('Basic Box Score Stats','MP') -> 'MP'; first col -> 'Player'
    cols = []
    for c in df.columns:
        if isinstance(c, tuple):
            cols.append(c[1] if c[1] and not str(c[1]).startswith("Unnamed") else c[0])
        else:
            cols.append(c)
    df.columns = cols
    df = df.rename(columns={df.columns[0]: "Player"})
    players, team_row = [], None
    for _, r in df.iterrows():
        name = str(r["Player"]).strip()
        if name in ("Reserves", "nan"):
            continue
        if name == "Team Totals":
            team_row = {STAT_MAP[k]: r.get(k) for k in STAT_MAP}
            continue
        mp = str(r.get("MP", "")).strip()
        row = {"player": name}
        # any non-time MP value is a DNP status: "Did Not Play",
        # "Did Not Dress", "Not With Team", etc.
        if not re.match(r"^\d+:\d{2}$", mp):
            row["mp"] = "DNP"
            for k in STAT_MAP:
                if k != "MP":
                    row[STAT_MAP[k]] = None
        else:
            for k, out in STAT_MAP.items():
                row[out] = r.get(k)
        players.append(row)
    return players, team_row


def num(x):
    if x is None:
        return None
    s = str(x).strip()
    if s in ("", "Did Not Play", "Did Not Dress"):
        return None
    try:
        return float(s)
    except ValueError:
        return None


In [ ]:
def main():
    odds = pd.read_csv(ODDS)
    urlmap = json.load(open(URLMAP))
    months = sorted(odds["date"].str.slice(0, 7).unique())
    failures = []
    if os.path.exists(FAIL_LOG):
        failures = json.load(open(FAIL_LOG))

    for month in months:
        pfile = PARTS / f'player_{month}.csv'
        tfile = PARTS / f'team_{month}.csv'
        if os.path.exists(pfile) and os.path.exists(tfile):
            print(f"{month}: already done, skipping", flush=True)
            continue
        games = odds[odds["date"].str.startswith(month)]
        prow, trow = [], []
        for _, g in games.iterrows():
            key = f"{g['date']}|{g['home_team']}"
            url = urlmap[key]
            try:
                html = fetch(url)
            except Exception as e:
                print(f"  FAIL fetch {g['date']} {g['away_team']} @ {g['home_team']}: {e!r}", flush=True)
                failures.append({"date": g["date"], "away": g["away_team"],
                                 "home": g["home_team"], "error": repr(e)})
                continue
            officials = parse_officials(html)
            for side, team in (("away", g["away_team"]), ("home", g["home_team"])):
                code = BR_CODE[team]
                try:
                    players, team_tot = parse_table(html, f"box-{code}-game-basic")
                except Exception as e:
                    print(f"  FAIL parse {g['date']} {team}: {e!r}", flush=True)
                    failures.append({"date": g["date"], "away": g["away_team"],
                                     "home": g["home_team"], "error": f"parse {team}: {e!r}"})
                    players, team_tot = [], None
                for p in players:
                    out = {"date": g["date"], "away_team": g["away_team"],
                           "home_team": g["home_team"], "team": team}
                    for c in PLAYER_COLS[4:]:
                        out[c] = p.get(c)
                    prow.append(out)
                if team_tot is not None:
                    out = {"date": g["date"], "away_team": g["away_team"],
                           "home_team": g["home_team"], "team": team,
                           "officials": officials}
                    for c in TEAM_COLS[4:-1]:
                        out[c] = team_tot.get(c)
                    trow.append(out)
                else:
                    print(f"  WARN no team totals {g['date']} {team}", flush=True)
            time.sleep(DELAY)
        pd.DataFrame(prow, columns=PLAYER_COLS).to_csv(pfile, index=False)
        pd.DataFrame(trow, columns=TEAM_COLS).to_csv(tfile, index=False)
        print(f"{month}: {len(games)} games, {len(prow)} player rows, {len(trow)} team rows", flush=True)
        json.dump(failures, open(FAIL_LOG, "w"), indent=1)

    # combine
    pfiles = sorted(PARTS / f for f in os.listdir(PARTS) if f.startswith("player_"))
    tfiles = sorted(PARTS / f for f in os.listdir(PARTS) if f.startswith("team_"))
    p = pd.concat([pd.read_csv(f) for f in pfiles], ignore_index=True)
    t = pd.concat([pd.read_csv(f) for f in tfiles], ignore_index=True)
    p.to_csv(RAW / 'boxscores_player.csv', index=False)
    t.to_csv(RAW / 'boxscores_team.csv', index=False)
    print(f"DONE: player rows={len(p)} team rows={len(t)} failures={len(failures)}", flush=True)
    print(f"games covered (team file): {t.groupby(['date','away_team','home_team']).ngroups}", flush=True)
    print(f"games with officials: {t.groupby(['date','away_team','home_team'])['officials'].first().notna().sum()} "
          f"of {t.groupby(['date','away_team','home_team']).ngroups}", flush=True)


In [ ]:
main()


## Retry pass

Retry pass for box-score games that failed in the main build.

The main build hit Basketball-Reference throttling in April 2026 (60 games,
all IncompleteRead mid-transfer after ~1,190 successful fetches). This script
re-fetches only those games, much more gently: 6s between requests, up to 5
attempts with exponential backoff, 60s timeout. Run after a cooldown pause.

Reads failures_boxscores.json, appends recovered rows to the 2026-04 part
files (deduped), then re-combines boxscores_player.csv / boxscores_team.csv.
Remaining failures go to failures_boxscores_retry.json.

The helpers above are already defined in this notebook, so the retry runs inline.


In [ ]:
RETRY_DELAY = 6.0

def fetch_gentle(url, timeout=60):
    last = None
    for attempt, wait in enumerate((0, 10, 20, 40, 80)):
        if wait:
            print(f"  backoff {wait}s", flush=True)
            time.sleep(wait)
        try:
            return fetch(url, timeout=timeout)
        except Exception as e:
            last = e
            print(f"  attempt {attempt + 1}/5 failed {url}: {e!r}", flush=True)
    raise last

def retry_main():
    failures = json.load(open(FAIL_LOG))
    print(f"{len(failures)} failed games to retry", flush=True)
    pfile = PARTS / 'player_2026-04.csv'
    tfile = PARTS / 'team_2026-04.csv'
    p_done = set()
    t_done = set()
    if os.path.exists(pfile):
        p_old = pd.read_csv(pfile)
        p_done = set(zip(p_old["date"], p_old["away_team"], p_old["home_team"], p_old["team"], p_old["player"]))
        p_teams = set(zip(p_old["date"], p_old["away_team"], p_old["home_team"], p_old["team"]))
    else:
        p_teams = set()
    if os.path.exists(tfile):
        t_old = pd.read_csv(tfile)
        t_done = set(zip(t_old["date"], t_old["away_team"], t_old["home_team"], t_old["team"]))
    prow, trow, still_failing = [], [], []
    for i, f in enumerate(failures):
        date, away, home = f["date"], f["away"], f["home"]
        key = (date, away, home)
        if all((date, away, home, tm) in p_teams for tm in (away, home)) and \
           all((date, away, home, tm) in t_done for tm in (away, home)):
            print(f"[{i + 1}/{len(failures)}] {date} {away} @ {home}: already present, skipping", flush=True)
            continue
        urlmap = json.load(open(URLMAP))
        url = urlmap[f"{date}|{home}"]
        try:
            html = fetch_gentle(url)
        except Exception as e:
            print(f"[{i + 1}/{len(failures)}] {date} {away} @ {home}: STILL FAIL_LOGING {e!r}", flush=True)
            still_failing.append({**f, "retry_error": repr(e)})
            continue
        officials = parse_officials(html)
        ok = True
        for team in (away, home):
            code = BR_CODE[team]
            try:
                players, team_tot = parse_table(html, f"box-{code}-game-basic")
            except Exception as e:
                print(f"  parse fail {date} {team}: {e!r}", flush=True)
                ok = False
                continue
            for p in players:
                if (date, away, home, team, p["player"]) in p_done:
                    continue
                out = {"date": date, "away_team": away, "home_team": home, "team": team}
                for c in PLAYER_COLS[4:]:
                    out[c] = p.get(c)
                prow.append(out)
            if team_tot is not None:
                out = {"date": date, "away_team": away, "home_team": home,
                       "team": team, "officials": officials}
                for c in TEAM_COLS[4:-1]:
                    out[c] = team_tot.get(c)
                trow.append(out)
        print(f"[{i + 1}/{len(failures)}] {date} {away} @ {home}: recovered", flush=True)
        time.sleep(RETRY_DELAY)

    if prow:
        pd.DataFrame(prow, columns=PLAYER_COLS).to_csv(pfile, mode="a", header=False, index=False)
    if trow:
        pd.DataFrame(trow, columns=TEAM_COLS).to_csv(tfile, mode="a", header=False, index=False)
    json.dump(still_failing, open(CACHE / 'failures_boxscores_retry.json', "w"), indent=1)

    # re-combine
    pfiles = sorted(PARTS / f for f in os.listdir(PARTS) if f.startswith("player_"))
    tfiles = sorted(PARTS / f for f in os.listdir(PARTS) if f.startswith("team_"))
    p = pd.concat([pd.read_csv(f) for f in pfiles], ignore_index=True)
    t = pd.concat([pd.read_csv(f) for f in tfiles], ignore_index=True)
    p = p.drop_duplicates(subset=["date", "away_team", "home_team", "team", "player"])
    t = t.drop_duplicates(subset=["date", "away_team", "home_team", "team"])
    p.to_csv(RAW / 'boxscores_player.csv', index=False)
    t.to_csv(RAW / 'boxscores_team.csv', index=False)
    print(f"RETRY DONE: recovered {len(trow)} team rows, {len(prow)} player rows; "
          f"still failing: {len(still_failing)}", flush=True)
    print(f"FINAL: player rows={len(p)} team rows={len(t)} "
          f"games={t.groupby(['date', 'away_team', 'home_team']).ngroups}", flush=True)


In [ ]:
retry_main()
